In [2]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
from pathlib import Path
import ctypes
import joblib
import numpy as np
import pandas as pd


# Ask Windows to render the Tk interface sharply on high-DPI displays.
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except (AttributeError, OSError):
    try:
        ctypes.windll.user32.SetProcessDPIAware()
    except (AttributeError, OSError):
        pass


# ============================================================
# MODEL FILE LOCATION
# ============================================================

MODEL_FILENAME = "CFDSST_SVR.joblib"

# The order and spelling must exactly match the SVR training code.
FEATURE_NAMES = [
    "L",
    "Do",
    "to",
    "Di",
    "ti",
    "fc",
    "fyo",
    "fyi",
]

# Simple professional colour palette.
BACKGROUND_COLOR = "#EEF3F7"
PANEL_COLOR = "#FFFFFF"
PRIMARY_COLOR = "#173F5F"
ACCENT_COLOR = "#247BA0"
LIGHT_ACCENT_COLOR = "#DCEAF2"
SUCCESS_COLOR = "#238636"
RESULT_COLOR = "#C0392B"


def find_model_file():
    """
    Search for the saved model in common locations.

    Search order:
    1. Folder containing this Python script
    2. Current working directory (recommended for Jupyter Notebook)
    3. Desktop
    4. OneDrive Desktop
    5. Manual file selection
    """

    possible_paths = [
        Path.cwd() / MODEL_FILENAME,
        Path.home() / "Desktop" / MODEL_FILENAME,
        Path.home() / "OneDrive" / "Desktop" / MODEL_FILENAME,
    ]

    # When running as a normal .py file, also check the script folder.
    try:
        script_folder = Path(__file__).resolve().parent
        possible_paths.insert(0, script_folder / MODEL_FILENAME)
    except NameError:
        # __file__ is unavailable inside Jupyter Notebook.
        pass

    # Return the first valid path.
    for path in possible_paths:
        if path.is_file():
            return path

    # If the model was not found, ask the user to select it.
    selected_file = filedialog.askopenfilename(
        title="Select the SVR-CFDSST model file",
        filetypes=[
            ("Joblib model", "*.joblib"),
            ("All files", "*.*"),
        ],
    )

    if selected_file:
        return Path(selected_file)

    return None


def load_model():
    """Find and load the trained SVR-CFDSST model."""

    model_file = find_model_file()

    if model_file is None:
        messagebox.showerror(
            "Model File Not Found",
            f"The model file '{MODEL_FILENAME}' could not be found.\n\n"
            "Please place it in the same folder as this notebook or "
            "Python file, and then run the program again.",
        )
        return None, None

    try:
        loaded_model = joblib.load(model_file)
        return loaded_model, model_file

    except Exception as error:
        messagebox.showerror(
            "Model Loading Error",
            f"The model file was found but could not be loaded:\n\n"
            f"{model_file}\n\n"
            f"Error details:\n{error}",
        )
        return None, model_file


# ============================================================
# GUI FUNCTIONS
# ============================================================

def predict():
    """Read the eight inputs and calculate the predicted axial capacity."""

    if model is None:
        messagebox.showerror(
            "Model Error",
            "The prediction model has not been loaded.",
        )
        return

    try:
        # Read and convert the input values in the exact training-feature order.
        input_values = [
            float(entry_x1.get().strip()),  # L
            float(entry_x2.get().strip()),  # Do
            float(entry_x3.get().strip()),  # to
            float(entry_x4.get().strip()),  # Di
            float(entry_x5.get().strip()),  # ti
            float(entry_x6.get().strip()),  # fc
            float(entry_x7.get().strip()),  # fyo
            float(entry_x8.get().strip()),  # fyi
        ]

        # All geometric and material inputs must be positive.
        if any(value <= 0 for value in input_values):
            raise ValueError("All input values must be greater than zero.")

        L, Do, to, Di, ti, fc, fyo, fyi = input_values

        # Basic CFDSST geometric-validity checks.
        if 2.0 * to >= Do:
            raise ValueError(
                "The outer-tube thickness is invalid: 2to must be less than Do."
            )

        if 2.0 * ti >= Di:
            raise ValueError(
                "The inner-tube thickness is invalid: 2ti must be less than Di."
            )

        if Di >= Do - 2.0 * to:
            raise ValueError(
                "The inner tube does not fit inside the outer tube: "
                "Di must be less than Do - 2to."
            )

        # Use a DataFrame because the trained ColumnTransformer selects inputs
        # by their exact column names.
        model_input = pd.DataFrame(
            [input_values],
            columns=FEATURE_NAMES,
        )

        # Make the prediction.
        prediction_result = model.predict(model_input)

        # MultiOutputRegressor returns a two-dimensional one-target output;
        # reshape safely to obtain one numerical prediction.
        prediction = float(
            np.asarray(prediction_result, dtype=float).reshape(-1)[0]
        )

        # Display the predicted ultimate axial capacity.
        output_label.config(text=f"{prediction:,.4f}")

    except ValueError as error:
        messagebox.showerror(
            "Input Error",
            "Please enter a valid numerical value for every input parameter.\n\n"
            f"Details:\n{error}",
        )

    except Exception as error:
        messagebox.showerror(
            "Prediction Error",
            f"The prediction could not be completed.\n\n"
            f"Error details:\n{error}",
        )


def clear_inputs():
    """Clear all input fields and the prediction result."""

    for entry in input_entries:
        entry.delete(0, tk.END)

    output_label.config(text="")
    entry_x1.focus_set()


# ============================================================
# CREATE THE MAIN WINDOW
# ============================================================

root = tk.Tk()
root.title("Circular CFDSST Capacity Prediction")
root.geometry("1460x820")
root.minsize(1240, 740)
root.configure(bg=BACKGROUND_COLOR)

# Keep the interface simple while applying one consistent colour theme.
style = ttk.Style(root)
try:
    style.theme_use("clam")
except tk.TclError:
    pass

style.configure(
    "TFrame",
    background=BACKGROUND_COLOR,
)
style.configure(
    "Panel.TLabelframe",
    background=PANEL_COLOR,
    bordercolor="#B8C6D1",
    relief="solid",
)
style.configure(
    "Panel.TLabelframe.Label",
    background=PANEL_COLOR,
    foreground=PRIMARY_COLOR,
    font=("Consolas", 11, "bold"),
)
style.configure(
    "Panel.TLabel",
    background=PANEL_COLOR,
    foreground="#17212B",
)
style.configure(
    "TEntry",
    fieldbackground="#FFFFFF",
    foreground="#17212B",
    bordercolor="#9FB2C1",
    lightcolor=ACCENT_COLOR,
    darkcolor="#9FB2C1",
)
style.configure(
    "Primary.TButton",
    font=("Consolas", 11, "bold"),
    foreground="white",
    background=ACCENT_COLOR,
    bordercolor=ACCENT_COLOR,
    padding=(18, 7),
)
style.map(
    "Primary.TButton",
    background=[("active", PRIMARY_COLOR)],
)
style.configure(
    "Simple.TButton",
    font=("Consolas", 11),
    foreground=PRIMARY_COLOR,
    background="#FFFFFF",
    bordercolor="#9FB2C1",
    padding=(18, 7),
)
style.map(
    "Simple.TButton",
    background=[("active", LIGHT_ACCENT_COLOR)],
)


# ============================================================
# LOAD THE MODEL
# ============================================================

model, model_path = load_model()


# ============================================================
# TITLE
# ============================================================

title_label = tk.Label(
    root,
    text="SVR Prediction of Circular CFDSST Columns",
    font=("Consolas", 18, "bold"),
    foreground="white",
    background=PRIMARY_COLOR,
    anchor="center",
    pady=12,
)

title_label.pack(
    fill="x",
    padx=0,
    pady=(0, 8),
)


# ============================================================
# MODEL STATUS
# ============================================================

if model is not None:
    model_status_text = f"Model loaded successfully: {model_path}"
    model_status_color = "green"
else:
    model_status_text = "Model not loaded"
    model_status_color = "red"

model_status_label = tk.Label(
    root,
    text=model_status_text,
    font=("Consolas", 10),
    foreground=model_status_color,
    background=BACKGROUND_COLOR,
    anchor="w",
)

model_status_label.pack(
    fill="x",
    padx=15,
    pady=(0, 5),
)


# ============================================================
# INPUT PARAMETERS
# ============================================================

input_frame = ttk.LabelFrame(
    root,
    text="Input Parameters",
    padding=(10, 10, 10, 10),
    style="Panel.TLabelframe",
)

input_frame.pack(
    padx=15,
    pady=10,
    fill="x",
)

input_frame.columnconfigure(0, weight=1)
input_frame.columnconfigure(1, weight=0)
input_frame.columnconfigure(2, weight=0)


# X1
ttk.Label(
    input_frame,
    text="X1: Column length (L, mm)",
    font=("Consolas", 14),
    style="Panel.TLabel",
).grid(row=0, column=0, padx=5, pady=5, sticky=tk.W)

entry_x1 = ttk.Entry(input_frame, font=("Consolas", 14), width=20)
entry_x1.grid(row=0, column=1, padx=5, pady=5)


# X2
ttk.Label(
    input_frame,
    text="X2: Outer diameter of outer steel tube (Do, mm)",
    font=("Consolas", 14),
    style="Panel.TLabel",
).grid(row=1, column=0, padx=5, pady=5, sticky=tk.W)

entry_x2 = ttk.Entry(input_frame, font=("Consolas", 14), width=20)
entry_x2.grid(row=1, column=1, padx=5, pady=5)


# X3
ttk.Label(
    input_frame,
    text="X3: Thickness of outer steel tube (to, mm)",
    font=("Consolas", 14),
    style="Panel.TLabel",
).grid(row=2, column=0, padx=5, pady=5, sticky=tk.W)

entry_x3 = ttk.Entry(input_frame, font=("Consolas", 14), width=20)
entry_x3.grid(row=2, column=1, padx=5, pady=5)


# X4
ttk.Label(
    input_frame,
    text="X4: Outer diameter of inner steel tube (Di, mm)",
    font=("Consolas", 14),
    style="Panel.TLabel",
).grid(row=3, column=0, padx=5, pady=5, sticky=tk.W)

entry_x4 = ttk.Entry(input_frame, font=("Consolas", 14), width=20)
entry_x4.grid(row=3, column=1, padx=5, pady=5)


# X5
ttk.Label(
    input_frame,
    text="X5: Thickness of inner steel tube (ti, mm)",
    font=("Consolas", 14),
    style="Panel.TLabel",
).grid(row=4, column=0, padx=5, pady=5, sticky=tk.W)

entry_x5 = ttk.Entry(input_frame, font=("Consolas", 14), width=20)
entry_x5.grid(row=4, column=1, padx=5, pady=5)


# X6
ttk.Label(
    input_frame,
    text="X6: Concrete compressive strength (fc, MPa)",
    font=("Consolas", 14),
    style="Panel.TLabel",
).grid(row=5, column=0, padx=5, pady=5, sticky=tk.W)

entry_x6 = ttk.Entry(input_frame, font=("Consolas", 14), width=20)
entry_x6.grid(row=5, column=1, padx=5, pady=5)


# X7
ttk.Label(
    input_frame,
    text="X7: Yield strength of outer steel tube (fyo, MPa)",
    font=("Consolas", 14),
    style="Panel.TLabel",
).grid(row=6, column=0, padx=5, pady=5, sticky=tk.W)

entry_x7 = ttk.Entry(input_frame, font=("Consolas", 14), width=20)
entry_x7.grid(row=6, column=1, padx=5, pady=5)


# X8
ttk.Label(
    input_frame,
    text="X8: Yield strength of inner steel tube (fyi, MPa)",
    font=("Consolas", 14),
    style="Panel.TLabel",
).grid(row=7, column=0, padx=5, pady=5, sticky=tk.W)

entry_x8 = ttk.Entry(input_frame, font=("Consolas", 14), width=20)
entry_x8.grid(row=7, column=1, padx=5, pady=5)


input_entries = [
    entry_x1,
    entry_x2,
    entry_x3,
    entry_x4,
    entry_x5,
    entry_x6,
    entry_x7,
    entry_x8,
]


# ============================================================
# HIGH-RESOLUTION COLUMN AND CROSS-SECTION SCHEMATIC
# ============================================================

schematic = tk.Canvas(
    input_frame,
    width=560,
    height=430,
    background=PANEL_COLOR,
    highlightthickness=1,
    highlightbackground=LIGHT_ACCENT_COLOR,
)
schematic.grid(
    row=0,
    column=2,
    rowspan=8,
    padx=(25, 5),
    pady=0,
    sticky="nsew",
)

schematic.create_text(
    280,
    18,
    text="Circular CFDSST geometry and axial loading",
    fill=PRIMARY_COLOR,
    font=("Consolas", 11, "bold"),
)

# ------------------------------------------------------------
# (a) Short-column elevation with axial loads and length L
# ------------------------------------------------------------
schematic.create_text(
    112, 45, text="(a) Loaded column", fill=PRIMARY_COLOR,
    font=("Times New Roman", 11, "italic"),
)

column_left, column_right = 78, 146
column_top, column_bottom = 112, 294

# Loading plates.
schematic.create_rectangle(63, 101, 161, 112, fill="#667681", outline=PRIMARY_COLOR, width=2)
schematic.create_rectangle(63, 294, 161, 305, fill="#667681", outline=PRIMARY_COLOR, width=2)

# Outer tube and concrete in elevation.
schematic.create_rectangle(
    column_left, column_top, column_right, column_bottom,
    fill="#D9E1E6", outline=PRIMARY_COLOR, width=5,
)
schematic.create_line(column_left + 10, column_top, column_left + 10, column_bottom,
                      fill=ACCENT_COLOR, width=3)
schematic.create_line(column_right - 10, column_top, column_right - 10, column_bottom,
                      fill=ACCENT_COLOR, width=3)
schematic.create_oval(column_left, 105, column_right, 119,
                      fill="#D9E1E6", outline=PRIMARY_COLOR, width=3)
schematic.create_oval(column_left, 287, column_right, 301,
                      fill="#D9E1E6", outline=PRIMARY_COLOR, width=3)

# Equal and opposite axial compressive loads P.
load_color = RESULT_COLOR
schematic.create_line(112, 55, 112, 97, fill=load_color, width=4,
                      arrow=tk.LAST, arrowshape=(14, 17, 7))
schematic.create_text(128, 70, text="P", fill=load_color,
                      font=("Times New Roman", 15, "bold italic"))
schematic.create_line(112, 352, 112, 310, fill=load_color, width=4,
                      arrow=tk.LAST, arrowshape=(14, 17, 7))
schematic.create_text(128, 340, text="P", fill=load_color,
                      font=("Times New Roman", 15, "bold italic"))

# Column-length dimension L with extension lines.
dim_color = "#8B1E1E"
schematic.create_line(48, column_top, 48, column_bottom, fill=dim_color, width=2.5,
                      arrow=tk.BOTH, arrowshape=(11, 13, 5))
schematic.create_line(48, column_top, column_left - 3, column_top,
                      fill=dim_color, width=1.5)
schematic.create_line(48, column_bottom, column_left - 3, column_bottom,
                      fill=dim_color, width=1.5)
schematic.create_text(34, (column_top + column_bottom) / 2, text="L",
                      fill=dim_color, font=("Times New Roman", 16, "bold italic"))

# ------------------------------------------------------------
# (b) Circular CFDSST cross-section
# ------------------------------------------------------------
schematic.create_text(
    370, 45, text="(b) Cross-section", fill=PRIMARY_COLOR,
    font=("Times New Roman", 11, "italic"),
)

cx, cy = 370, 190
outer_radius = 82
outer_clear_radius = 68
inner_outer_radius = 39
inner_clear_radius = 27

# Outer stainless-steel tube.
schematic.create_oval(
    cx - outer_radius,
    cy - outer_radius,
    cx + outer_radius,
    cy + outer_radius,
    fill=PRIMARY_COLOR,
    outline=PRIMARY_COLOR,
    width=2,
)

# Concrete annulus.
schematic.create_oval(
    cx - outer_clear_radius,
    cy - outer_clear_radius,
    cx + outer_clear_radius,
    cy + outer_clear_radius,
    fill="#D9E1E6",
    outline=PRIMARY_COLOR,
    width=1,
)

# Inner steel tube.
schematic.create_oval(
    cx - inner_outer_radius,
    cy - inner_outer_radius,
    cx + inner_outer_radius,
    cy + inner_outer_radius,
    fill=ACCENT_COLOR,
    outline=ACCENT_COLOR,
    width=2,
)
schematic.create_oval(
    cx - inner_clear_radius,
    cy - inner_clear_radius,
    cx + inner_clear_radius,
    cy + inner_clear_radius,
    fill="#FFFFFF",
    outline=ACCENT_COLOR,
    width=1,
)

# Centre lines, drawn lightly behind the dimensions.
schematic.create_line(
    cx - outer_radius - 8,
    cy,
    cx + outer_radius + 8,
    cy,
    fill="#7A8791",
    dash=(3, 3),
    width=1,
)
schematic.create_line(
    cx,
    cy - outer_radius - 8,
    cx,
    cy + outer_radius + 8,
    fill="#7A8791",
    dash=(3, 3),
    width=1,
)

# Outer diameter D_o.
dimension_y = 80
schematic.create_line(
    cx - outer_radius,
    dimension_y,
    cx + outer_radius,
    dimension_y,
    fill=RESULT_COLOR,
    width=2.5,
    arrow=tk.BOTH,
    arrowshape=(8, 10, 4),
)
schematic.create_line(
    cx - outer_radius, dimension_y, cx - outer_radius, cy - outer_radius,
    fill=RESULT_COLOR, width=1,
)
schematic.create_line(
    cx + outer_radius, dimension_y, cx + outer_radius, cy - outer_radius,
    fill=RESULT_COLOR, width=1,
)
schematic.create_text(
    cx,
    63,
    text="Dₒ",
    fill=RESULT_COLOR,
    font=("Times New Roman", 15, "bold italic"),
)

# Inner-tube outer diameter D_i. The dimension line is completely below
# the section so it cannot cover the concrete or inner tube.
inner_dimension_y = cy + outer_radius + 23
schematic.create_line(
    cx - inner_outer_radius,
    inner_dimension_y,
    cx + inner_outer_radius,
    inner_dimension_y,
    fill=RESULT_COLOR,
    width=2.5,
    arrow=tk.BOTH,
    arrowshape=(9, 11, 5),
)
schematic.create_line(cx - inner_outer_radius, cy + inner_outer_radius,
                      cx - inner_outer_radius, inner_dimension_y,
                      fill=RESULT_COLOR, width=1.5)
schematic.create_line(cx + inner_outer_radius, cy + inner_outer_radius,
                      cx + inner_outer_radius, inner_dimension_y,
                      fill=RESULT_COLOR, width=1.5)
schematic.create_text(
    cx,
    inner_dimension_y + 17,
    text="Dᵢ",
    fill=RESULT_COLOR,
    font=("Times New Roman", 14, "bold italic"),
)

# Outer-tube thickness t_o. Use an external leader so the label does not
# obscure the cross-section.
schematic.create_line(
    cx + 58,
    cy - 49,
    cx + 105,
    cy - 88,
    fill="#F39C12",
    width=2.5,
    arrow=tk.FIRST,
    arrowshape=(10, 12, 5),
)
schematic.create_text(
    cx + 116,
    cy - 94,
    text="tₒ",
    fill="#B56B00",
    font=("Times New Roman", 14, "bold italic"),
)

# Inner-tube thickness t_i, also shown with an external leader.
schematic.create_line(
    cx + 28,
    cy - 26,
    cx + 108,
    cy - 18,
    fill="#F39C12",
    width=2.5,
    arrow=tk.FIRST,
    arrowshape=(10, 12, 5),
)
schematic.create_text(
    cx + 121,
    cy - 18,
    text="tᵢ",
    fill="#B56B00",
    font=("Times New Roman", 14, "bold italic"),
)

# Clear material legend using mathematical symbols.
legend_y = 347
schematic.create_rectangle(243, legend_y, 258, legend_y + 15,
                           fill=PRIMARY_COLOR, outline=PRIMARY_COLOR)
schematic.create_text(265, legend_y + 8, text="Outer steel, fᵧₒ", anchor="w",
                      fill="#17212B", font=("Times New Roman", 10, "italic"))
schematic.create_rectangle(376, legend_y, 391, legend_y + 15,
                           fill=ACCENT_COLOR, outline=ACCENT_COLOR)
schematic.create_text(398, legend_y + 8, text="Inner steel, fᵧᵢ", anchor="w",
                      fill="#17212B", font=("Times New Roman", 10, "italic"))
schematic.create_rectangle(243, legend_y + 25, 258, legend_y + 40,
                           fill="#D9E1E6", outline=PRIMARY_COLOR)
schematic.create_text(265, legend_y + 33, text="Concrete, fᶜ", anchor="w",
                      fill="#17212B", font=("Times New Roman", 10, "italic"))
schematic.create_text(
    370, 415,
    text="Dimensions:  L,  Dₒ,  tₒ,  Dᵢ,  tᵢ",
    fill=PRIMARY_COLOR,
    font=("Times New Roman", 11, "bold italic"),
)


# ============================================================
# PREDICTION RESULT
# ============================================================

output_frame = ttk.LabelFrame(
    root,
    text="Prediction Result",
    padding=(10, 10, 10, 10),
    style="Panel.TLabelframe",
)

output_frame.pack(
    padx=15,
    pady=10,
    fill="x",
)

ttk.Label(
    output_frame,
    text="Predicted Ultimate Axial Capacity (Nu, MN) =",
    font=("Consolas", 14),
    style="Panel.TLabel",
).grid(row=0, column=0, padx=5, pady=5, sticky=tk.W)

output_label = ttk.Label(
    output_frame,
    text="",
    font=("Consolas", 14, "bold"),
    foreground=RESULT_COLOR,
    background=PANEL_COLOR,
)

output_label.grid(
    row=0,
    column=1,
    padx=5,
    pady=5,
    sticky=tk.W,
)


# ============================================================
# BUTTONS
# ============================================================

buttons_frame = ttk.Frame(root)
buttons_frame.pack(pady=10)

predict_button = ttk.Button(
    buttons_frame,
    text="Predict",
    command=predict,
    style="Primary.TButton",
)
predict_button.grid(row=0, column=0, padx=8)

clear_button = ttk.Button(
    buttons_frame,
    text="Clear",
    command=clear_inputs,
    style="Simple.TButton",
)
clear_button.grid(row=0, column=1, padx=8)

exit_button = ttk.Button(
    buttons_frame,
    text="Exit",
    command=root.destroy,
    style="Simple.TButton",
)
exit_button.grid(row=0, column=2, padx=8)


# Allow prediction by pressing Enter.
root.bind("<Return>", lambda event: predict())

# Place the cursor in the first input field.
entry_x1.focus_set()

# Start the GUI event loop.
root.mainloop()
